# Maricopa Valley Fever — corrected 2015–2025 modeling

**Input:** `Maricopa_2015_2025_QC_Corrected_Model.xlsx`.

This notebook uses the fully recovered five-station AZMet data and fixes invalid `999` dew-point readings. It **does not download AZMet weather again** and **does not alter ADHS case counts**. The data span 2015–2025. Evaluate on 2024 as historical held-out data, and interpret 2025 cautiously because the case numbers are provisional. The model forecasts **next month** from the previous month's data.


In [ ]:
import pandas as pd, numpy as np
from google.colab import files
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from pathlib import Path
print("Upload: Maricopa_2015_2025_QC_Corrected_Model.xlsx")
up=files.upload()
name=next((x for x in up if x.lower().endswith('.xlsx')), None)
if name is None: raise ValueError("Upload the corrected .xlsx model input")
fm=pd.read_excel(name,sheet_name='FORECAST_READY')
cm=pd.read_excel(name,sheet_name='COUNTY_MONTH')
print('Monthly rows:',len(cm),'Forecast rows:',len(fm),'AZMet stations:',5)
print('Date coverage:',cm.Month.min(),'to',cm.Month.max())
assert len(cm)==132 and len(fm)==131
assert cm[cm.Month.str[:4]=='2015'].VF_cases_event_date.sum()==5348
assert cm[cm.Month.str[:4]=='2024'].VF_cases_event_date.sum()==10920
assert cm[cm.Month.str[:4]=='2025'].VF_cases_event_date.sum()==8548
assert np.isclose(cm.loc[cm.Month.eq('2016-08'),'AZMET_dewpoint_mean_C'].iloc[0],15.53096774193548,atol=0.01)
print('Case totals and invalid-dewpoint correction checks: PASS')


In [ ]:
CASE=['cases_current','cases_lag1','cases_same_target_month_last_year']
MODELS={
 'Case history Ridge': CASE,
 'Case + PM10 Ridge': CASE+['EPA_PM10_monthly_mean_ugm3','EPA_PM10_daily_p95_ugm3'],
 'Case + rain Ridge': CASE+['PRISM_rain_total_in','rain_prev3m_in'],
 'Case + AZMet Ridge': CASE+['AZMET_soil50_mean_C','AZMET_wind_mean_mph','AZMET_humidity_mean_pct'],
 'Case + wind vectors + VPD Ridge': CASE+['AZMET_wind_u_mean_mps','AZMET_wind_v_mean_mps','AZMET_vpd_mean_kPa'],
 'Case + PM10 + rain + soil Ridge': CASE+['EPA_PM10_monthly_mean_ugm3','PRISM_rain_total_in','AZMET_soil50_mean_C'],
 'Case + PM10 + rain + soil + wind + VPD Ridge':CASE+['EPA_PM10_monthly_mean_ugm3','PRISM_rain_total_in','AZMET_soil50_mean_C','AZMET_wind_u_mean_mps','AZMET_wind_v_mean_mps','AZMET_vpd_mean_kPa'],
}
TARGET='target_next_month_VF_cases'
NEEDED=sorted(set(sum(MODELS.values(),[])+[TARGET]))
good=fm[fm.all_sources_complete.astype(bool)].dropna(subset=NEEDED).copy()
good=good.sort_values('target_month').reset_index(drop=True)
print(good.groupby('split').size().to_string())
print('Available rows for modeling:',len(good),'with complete feature sets')

def predict_one(train,test,model_name):
 if model_name=='Seasonal baseline':return test['cases_same_target_month_last_year'].to_numpy(float)
 feature_cols=MODELS[model_name]
 alg=make_pipeline(StandardScaler(),Ridge(alpha=10.0))
 alg.fit(train[feature_cols],train[TARGET])
 return alg.predict(test[feature_cols])

def metric_rows(pred_df,label):
 out=[]
 for m in ['Seasonal baseline']+list(MODELS):
  out.append({'Evaluation':label,'Model':m,'Months':len(pred_df),'MAE':mean_absolute_error(pred_df.Actual,pred_df[m]), 'RMSE':np.sqrt(mean_squared_error(pred_df.Actual,pred_df[m]))})
 return pd.DataFrame(out).sort_values('MAE').reset_index(drop=True)


In [ ]:
# Expanding-window validation ends with 2023. Keep all 2024 and 2025 out of this loop.
pre=good[good.target_year<=2023].reset_index(drop=True)
preds=[]
for i in range(24,len(pre)):
 tr,te=pre.iloc[:i],pre.iloc[[i]]
 r={'target_month':te.target_month.iloc[0],'Actual':float(te[TARGET].iloc[0])}
 for m in ['Seasonal baseline']+list(MODELS):r[m]=float(predict_one(tr,te,m)[0])
 preds.append(r)
walk_preds=pd.DataFrame(preds)
walk_metrics=metric_rows(walk_preds,'Walk-forward, through 2023')
print('Walk-forward results (no 2024 or 2025 model training):')
display(walk_metrics)


In [ ]:
# Fixed held-out 2024; no 2024 rows in training.
held=good[good.target_year==2024].copy()
tr=good[good.target_year<=2023].copy()
def evaluate_holdout(held,train,label):
 pred=pd.DataFrame({'target_month':held.target_month.to_numpy(),'Actual':held[TARGET].to_numpy(float)})
 for m in ['Seasonal baseline']+list(MODELS): pred[m]=predict_one(train,held,m)
 return pred,metric_rows(pred,label)
held_pred,held_metrics=evaluate_holdout(held,tr,'Held-out 2024')
print('2024 held-out results (investigate January–March surveillance anomaly):')
display(held_metrics)


In [ ]:
# Optional 2025 retrospective only: numbers may be revised.
held25=good[good.target_year==2025].copy()
tr25=good[good.target_year<=2024].copy()
provisional_pred,provisional_metrics=evaluate_holdout(held25,tr25,'2025 provisional')
print('2025 provisional comparisons — not a final benchmark:')
display(provisional_metrics)


In [ ]:
# Output results. No target-month weather or case values used as model features.
walk_metrics.to_csv('maricopa_walkforward_metrics.csv',index=False)
held_metrics.to_csv('maricopa_2024_heldout_metrics.csv',index=False)
provisional_metrics.to_csv('maricopa_2025_provisional_metrics.csv',index=False)
walk_preds.to_csv('maricopa_walkforward_predictions.csv',index=False)
held_pred.to_csv('maricopa_2024_heldout_predictions.csv',index=False)
provisional_pred.to_csv('maricopa_2025_provisional_predictions.csv',index=False)
import zipfile
with zipfile.ZipFile('maricopa_reviewed_model_results.zip','w',compression=zipfile.ZIP_DEFLATED) as z:
 for p in Path('.').glob('maricopa_*.csv'):z.write(p)
files.download('maricopa_reviewed_model_results.zip')
